In [9]:
import pandas as pd
import numpy as np
import matplotlib as plt

data_main = pd.read_csv('sources/upload-weeklyshipcrossingsforsixmaritimepassagesofinterest.csv')
data_excel = pd.read_excel(
    'sources/uktradeflowsofcontainerisedproductsthroughglobalmaritimepassages20202024.xlsx',
    sheet_name=['1.Monthly Volumes All (Imports)', '2.Monthly Volumes All (Exports)']
)
data_second = pd.concat(data_excel.values(), ignore_index=True)
data_war = pd.read_csv('sources/GEDEvent_v25_1.csv')

C:\Users\rakad\AppData\Local\Temp\ipykernel_21272\360238018.py:11: DtypeWarning: Columns (47) have mixed types. Specify dtype option on import or set low_memory=False.
  data_war = pd.read_csv('sources/GEDEvent_v25_1.csv')


In [10]:
data_main.head()

,Passage,Week of entry,Number of crossings
0,Bab-Al Mandab Strait,03/01/2022,394
1,Bab-Al Mandab Strait,10/01/2022,391
2,Bab-Al Mandab Strait,17/01/2022,386
3,Bab-Al Mandab Strait,24/01/2022,394
4,Bab-Al Mandab Strait,31/01/2022,415


In [11]:
data_main.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 695 entries, 0 to 694
Data columns (total 3 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   Passage              695 non-null    object
 1   Week of entry        695 non-null    object
 2   Number of crossings  695 non-null    int64 
dtypes: int64(1), object(2)
memory usage: 16.4+ KB


In [12]:
data_war.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 385918 entries, 0 to 385917
Data columns (total 49 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   id                 385918 non-null  int64  
 1   relid              385918 non-null  object 
 2   year               385918 non-null  int64  
 3   active_year        385918 non-null  bool   
 4   code_status        385918 non-null  object 
 5   type_of_violence   385918 non-null  int64  
 6   conflict_dset_id   385918 non-null  int64  
 7   conflict_new_id    385918 non-null  int64  
 8   conflict_name      385918 non-null  object 
 9   dyad_dset_id       385918 non-null  int64  
 10  dyad_new_id        385918 non-null  int64  
 11  dyad_name          385918 non-null  object 
 12  side_a_dset_id     385918 non-null  int64  
 13  side_a_new_id      385918 non-null  int64  
 14  side_a             385918 non-null  object 
 15  side_b_dset_id     385918 non-null  int64  
 16  si

In [13]:
data_second.head()

,Passage,Direction,TEU,Year,Month
0,Taiwan Strait,import,13625.1,2020,1
1,Taiwan Strait,import,2710.4,2020,2
2,Taiwan Strait,import,11166.5,2020,3
3,Taiwan Strait,import,7766,2020,4
4,Taiwan Strait,import,11118.4,2020,5


In [14]:
data_second.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 600 entries, 0 to 599
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   Passage    600 non-null    object
 1   Direction  600 non-null    object
 2   TEU        600 non-null    object
 3   Year       600 non-null    int64 
 4   Month      600 non-null    int64 
dtypes: int64(2), object(3)
memory usage: 23.6+ KB


##### Processing data_main

In [15]:
# Convert "Week of entry" to datetime (day/month/year format)
data_main['Week of entry'] = pd.to_datetime(data_main['Week of entry'], format='%d/%m/%Y')

data_main.head(10)


,Passage,Week of entry,Number of crossings
0,Bab-Al Mandab Strait,2022-01-03,394
1,Bab-Al Mandab Strait,2022-01-10,391
2,Bab-Al Mandab Strait,2022-01-17,386
3,Bab-Al Mandab Strait,2022-01-24,394
4,Bab-Al Mandab Strait,2022-01-31,415
5,Bab-Al Mandab Strait,2022-02-07,443
6,Bab-Al Mandab Strait,2022-02-14,364
7,Bab-Al Mandab Strait,2022-02-21,424
8,Bab-Al Mandab Strait,2022-02-28,394
9,Bab-Al Mandab Strait,2022-03-07,418


In [16]:
data_main.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 695 entries, 0 to 694
Data columns (total 3 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   Passage              695 non-null    object        
 1   Week of entry        695 non-null    datetime64[ns]
 2   Number of crossings  695 non-null    int64         
dtypes: datetime64[ns](1), int64(1), object(1)
memory usage: 16.4+ KB


##### Processing data_second

In [17]:
# 1. Replace "x" in TEU with NaN, then convert to numeric
data_second['TEU'] = pd.to_numeric(data_second['TEU'].replace('x', np.nan))

# 2. Combine Year and Month into a datetime column
data_second['Date'] = pd.to_datetime(data_second[['Year', 'Month']].assign(day=1))

# 3. Group by Passage and Month, summing TEU
data_second = data_second.groupby(
    ['Passage', data_second['Date'].dt.to_period('M')]
).agg({'TEU': 'sum'}).reset_index()

data_second.head(10)

C:\Users\rakad\AppData\Local\Temp\ipykernel_21272\3138249727.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data_second['TEU'] = pd.to_numeric(data_second['TEU'].replace('x', np.nan))


,Passage,Date,TEU
0,Cape of Good Hope,2020-01,511.2
1,Cape of Good Hope,2020-02,334.7
2,Cape of Good Hope,2020-03,730.3
3,Cape of Good Hope,2020-04,1493.4
4,Cape of Good Hope,2020-05,4806.4
5,Cape of Good Hope,2020-06,2669.0
6,Cape of Good Hope,2020-07,5794.9
7,Cape of Good Hope,2020-08,4294.9
8,Cape of Good Hope,2020-09,1857.6
9,Cape of Good Hope,2020-10,1045.2


##### Processing data_war

In [18]:
# 1. Filter years 2020-2024
data_war = data_war[(data_war['year'] >= 2020) & (data_war['year'] <= 2024)]

# 2. Keep only useful features
useful_columns = [
    'year', 'type_of_violence', 'conflict_name',
    'side_a', 'side_b',
    'country', 'region', 'adm_1',
    'latitude', 'longitude',
    'date_start', 'date_end',
    'deaths_a', 'deaths_b', 'deaths_civilians', 'deaths_unknown',
    'best', 'high', 'low'
]

data_war = data_war[useful_columns]

In [19]:
# Filter for Government of Israel and Hamas on either side
data_war = data_war[
    ((data_war['side_a'] == 'Government of Israel') & (data_war['side_b'] == 'Hamas')) |
    ((data_war['side_a'] == 'Hamas') & (data_war['side_b'] == 'Government of Israel'))
]

In [20]:
# Convert date_start to datetime and create year_month column
data_war['date_start'] = pd.to_datetime(data_war['date_start'])
data_war['year_month'] = data_war['date_start'].dt.to_period('M')

# Group by month and sum total deaths, keeping side_a and side_b
monthly_deaths = data_war.groupby(['year_month', 'side_a', 'side_b']).agg(
    deaths_a=('deaths_a', 'sum'),
    deaths_b=('deaths_b', 'sum'),
    deaths_civilians=('deaths_civilians', 'sum'),
    deaths_unknown=('deaths_unknown', 'sum'),
    best=('best', 'sum'),
    high=('high', 'sum'),
    low=('low', 'sum')
).reset_index()

monthly_deaths


,year_month,side_a,side_b,deaths_a,deaths_b,deaths_civilians,deaths_unknown,best,high,low
0,2020-01,Government of Israel,Hamas,0,0,0,3,3,3,3
1,2021-05,Government of Israel,Hamas,1,67,153,1,222,222,222
2,2021-08,Government of Israel,Hamas,1,2,1,1,5,5,5
3,2021-09,Government of Israel,Hamas,0,3,0,0,3,3,3
4,2021-11,Government of Israel,Hamas,0,1,1,0,2,2,2
5,2021-12,Government of Israel,Hamas,0,1,0,0,1,1,1
6,2022-01,Government of Israel,Hamas,2,0,0,0,2,2,2
7,2022-04,Government of Israel,Hamas,1,0,0,0,1,1,1
8,2022-10,Government of Israel,Hamas,0,2,2,1,5,5,5
9,2022-11,Government of Israel,Hamas,0,0,2,0,2,2,2


##### Save data

In [22]:
data_main.to_csv('sources/data_main.csv', index=False)
data_second.to_csv('sources/data_second.csv', index=False)
monthly_deaths.to_csv('sources/data_war.csv', index=False)